# 01 - Time of Day Analytics

Average cumulative USD gifts throughout the time of day, filtered by `tiktok_username`, with an interactive zoomable chart.

In [15]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import plotly.express as px

sns.set_theme(style='whitegrid', context='talk')

root = Path.cwd().resolve()
for candidate in [root, *root.parents]:
    if (candidate / 'analytics' / 'notebook_utils.py').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Could not locate repo root containing analytics/notebook_utils.py')

sys.path.insert(0, str(repo_root / 'analytics'))
import notebook_utils as nb

client = nb.get_client()
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 100)

In [16]:
def fetch_table_all(client, table, columns='*', page_size=1000, max_rows=100000, order_by='iso_ts', ascending=True):
    rows = []
    offset = 0
    while offset < max_rows:
        end = min(offset + page_size - 1, max_rows - 1)
        query = client.table(table).select(columns).range(offset, end)
        if order_by:
            query = query.order(order_by, desc=not ascending)
        batch = query.execute().data or []
        if not batch:
            break
        rows.extend(batch)
        if len(batch) < page_size:
            break
        offset += page_size
    return pd.DataFrame(rows)

gift_df = fetch_table_all(client, 'gift_events', page_size=1000, max_rows=100000)
print(f'Loaded {len(gift_df):,} gift_events rows.')
display(gift_df.head())

Loaded 5,448 gift_events rows.


,id,iso_ts,unix_ts,event_type,room_id,create_time_ms,message_id,gift_id,gift_name,diamond_count,repeat_count,combo_count,amount_value,fan_ticket_count,room_fan_ticket_count,group_count,repeat_end,from_user_id,from_username,from_nickname,to_user_id,to_username,to_nickname,to_member_id_int,to_member_nickname,anchor_id,send_type,order_id,group_id,describe,is_gift_giver_of_anchor,is_subscriber_of_anchor,is_mutual_following_with_anchor,is_follower_of_anchor,gift_monitor_from_platform,gift_monitor_from_version,gift_monitor_send_msg_ms,priority,room_message_heat_level,tiktok_username,gift_monitor_anchor_id,gift_monitor_to_user_id,gift_monitor_send_message_success_ms
0,1770627641285121,2026-02-09T09:00:43.441076+00:00,1770627643,gift,7604744834994621202,1770627640589,7604787474405198600,5487,Finger Heart,5,1,1,5,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.0,1,6975523201414480901,rahmatmzk_,رحمتى مىرزكى,7475077605381964808,moonstardance2,the host 1,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,7475077605381964808,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.770628e+12,رحمتى مىرزكى: gifted the host 1 Finger Heart,1,0,0,0,android,430803,1770627640590,"GiftImPriority(queue_sizes=[-1, -1, 50, 150], ...",2,moonstardance2,7.475078e+18,7.475078e+18,1.770628e+12
1,1770627661222127,2026-02-09T09:01:03.360824+00:00,1770627663,gift,7604744834994621202,1770627660287,7604787654253300487,5655,Rose,1,4,4,1,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.0,1,7544328249275712519,labubu5030,GBU,7475077605381964808,moonstardance2,the host,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,7475077605381964808,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.770628e+12,GBU: gifted the host 4 Rose,1,0,0,0,android,430703,1770627660287,"GiftImPriority(queue_sizes=[-1, -1, 50, 150], ...",3,moonstardance2,7.475078e+18,7.475078e+18,1.770628e+12
2,1770627665387129,2026-02-09T09:01:07.52529+00:00,1770627667,gift,7604744834994621202,1770627664588,7604787462134106887,5655,Rose,1,1,1,1,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.0,1,7544328249275712519,labubu5030,GBU,7475077605381964808,moonstardance2,the host,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,7475077605381964808,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.770628e+12,GBU: gifted the host 1 Rose,1,0,0,0,android,430703,1770627664589,"GiftImPriority(queue_sizes=[-1, -1, 50, 150], ...",3,moonstardance2,7.475078e+18,7.475078e+18,1.770628e+12
3,1770627676849141,2026-02-09T09:01:18.988703+00:00,1770627678,gift,7604744834994621202,1770627675760,7604787613669575444,5655,Rose,1,10,10,1,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.0,1,7544328249275712519,labubu5030,GBU,7475077605381964808,moonstardance2,the host,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,7475077605381964808,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.770628e+12,GBU: gifted the host 10 Rose,1,0,0,0,android,430703,1770627675760,"GiftImPriority(queue_sizes=[-1, -1, 50, 150], ...",3,moonstardance2,7.475078e+18,7.475078e+18,1.770628e+12
4,1770627703490154,2026-02-09T09:01:45.631141+00:00,1770627705,gift,7604744834994621202,1770627702945,7604787716454419220,5780,Bouquet Flower,30,3,3,30,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.0,1,6880433596177433602,abemung1,محمد شریف الدین بن رحيم,7475077605381964808,moonstardance2,the host 3,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,7475077605381964808,<object object at 0x00000239F6D65AA0>,<object object at 0x00000239F6D65AA0>,1.770628e+12,محمد شریف الدین بن رحيم: gifted the host 3 Bou...,1,0,0,1,android,430703,1770627702946,"GiftImPriority(queue_sizes=[-1, -1, 50, 150], ...",4

In [17]:
def normalize_group_id(series: pd.Series) -> pd.Series:
    return (
        series.astype('string')
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
        .replace({'<NA>': pd.NA, 'nan': pd.NA, 'None': pd.NA, '': pd.NA})
    )

for column_name in ['group_id', 'tiktok_username', 'iso_ts', 'diamond_count', 'repeat_count']:
    if column_name not in gift_df.columns:
        gift_df[column_name] = pd.NA

gift_df['group_id_norm'] = normalize_group_id(gift_df['group_id'])
gift_df['tiktok_username'] = gift_df['tiktok_username'].astype('string')

# Filters
TIKTOK_USERNAME = "viiceversa_ns"  # Example: 'visiondance.leo'
START_TS = None         # Example: '2026-02-01'
END_TS = None           # Example: '2026-02-15'

group_source = gift_df.copy()
if TIKTOK_USERNAME:
    group_source = group_source[group_source['tiktok_username'].str.lower() == str(TIKTOK_USERNAME).lower()]

top_groups = (
    group_source.dropna(subset=['group_id_norm'])
    .groupby('group_id_norm', dropna=True)
    .size()
    .sort_values(ascending=False)
    .head(20)
    .rename('events')
    .reset_index()
)
display(top_groups)

filtered = gift_df.copy()
if TIKTOK_USERNAME:
    filtered = filtered[filtered['tiktok_username'].str.lower() == str(TIKTOK_USERNAME).lower()]

filtered['iso_ts'] = pd.to_datetime(filtered['iso_ts'], errors='coerce', utc=True)
if START_TS:
    filtered = filtered[filtered['iso_ts'] >= pd.Timestamp(START_TS, tz='UTC')]
if END_TS:
    filtered = filtered[filtered['iso_ts'] < (pd.Timestamp(END_TS, tz='UTC') + pd.Timedelta(days=1))]

print(f'Rows after filters: {len(filtered):,}')

,group_id_norm,events
0,1770870057421,16
1,1770958710984,14
2,1770870038022,10
3,1770956682742,8
4,1770695507976,6
5,1770867342561,6
6,1770958393303,6
7,1770957387477,5
8,1770956314715,5
9,1770956841932,5


Rows after filters: 1,704


In [18]:
working = filtered.copy()
working = working.dropna(subset=['iso_ts'])
working['diamond_count'] = pd.to_numeric(working['diamond_count'], errors='coerce').fillna(0)
working['repeat_count'] = pd.to_numeric(working['repeat_count'], errors='coerce').fillna(0)
working['usd'] = working['diamond_count'] * working['repeat_count'] * 0.005
working['day'] = working['iso_ts'].dt.floor('D')
working['minute_of_day'] = working['iso_ts'].dt.hour * 60 + working['iso_ts'].dt.minute

minute_usd = (
    working.groupby(['day', 'minute_of_day'], as_index=False)['usd']
    .sum()
    .sort_values(['day', 'minute_of_day'])
)

if minute_usd.empty:
    time_of_day_avg = pd.DataFrame(columns=['minute_of_day', 'avg_cumulative_usd', 'time'])
else:
    all_days = minute_usd['day'].drop_duplicates().sort_values()
    minute_grid = pd.MultiIndex.from_product(
        [all_days, range(1440)],
        names=['day', 'minute_of_day'],
    ).to_frame(index=False)

    minute_usd = minute_grid.merge(minute_usd, on=['day', 'minute_of_day'], how='left')
    minute_usd['usd'] = minute_usd['usd'].fillna(0.0)
    minute_usd['cumulative_usd'] = minute_usd.groupby('day')['usd'].cumsum()

    time_of_day_avg = (
        minute_usd.groupby('minute_of_day', as_index=False)['cumulative_usd']
        .mean()
        .rename(columns={'cumulative_usd': 'avg_cumulative_usd'})
        .sort_values('minute_of_day')
    )
    time_of_day_avg['avg_cumulative_usd'] = time_of_day_avg['avg_cumulative_usd'].cummax()
    time_of_day_avg['time'] = pd.Timestamp('2000-01-01', tz='UTC') + pd.to_timedelta(time_of_day_avg['minute_of_day'], unit='m')

display(time_of_day_avg[['time', 'avg_cumulative_usd']].head(20))

,time,avg_cumulative_usd
0,2000-01-01 00:00:00+00:00,0.0
1,2000-01-01 00:01:00+00:00,0.0
2,2000-01-01 00:02:00+00:00,0.0
3,2000-01-01 00:03:00+00:00,0.0
4,2000-01-01 00:04:00+00:00,0.0
5,2000-01-01 00:05:00+00:00,0.0
6,2000-01-01 00:06:00+00:00,0.0
7,2000-01-01 00:07:00+00:00,0.0
8,2000-01-01 00:08:00+00:00,0.0
9,2000-01-01 00:09:00+00:00,0.0


In [19]:
if time_of_day_avg.empty:
    print('No rows matched the selected filters.')
else:
    label_user = str(TIKTOK_USERNAME) if TIKTOK_USERNAME else 'all users'
    title = f'Average Cumulative USD Gifts by Time of Day ({label_user})'
    plot_df = time_of_day_avg.copy()
    line_color = sns.color_palette('crest', 1).as_hex()[0]

    fig = px.line(
        plot_df,
        x='time',
        y='avg_cumulative_usd',
        title=title,
        template='plotly_white',
    )
    fig.update_traces(line={'color': line_color, 'width': 3})
    fig.update_layout(
        xaxis_title='Time of Day (UTC)',
        yaxis_title='Average Cumulative USD',
        hovermode='x unified',
    )
    fig.show()

In [20]:
if working.empty:
    print('No rows matched the selected filters.')
else:
    USD_CHANGE_WINDOW_MINUTES = 30

    # Change in cumulative USD at each minute is the per-minute USD increment.
    usd_change_hist = (
        minute_usd.groupby('minute_of_day', as_index=False)['usd']
        .mean()
        .rename(columns={'usd': 'avg_delta_cumulative_usd'})
        .sort_values('minute_of_day')
    )
    usd_change_hist['avg_delta_cumulative_usd_smoothed'] = (
        usd_change_hist['avg_delta_cumulative_usd']
        .rolling(window=USD_CHANGE_WINDOW_MINUTES, center=True, min_periods=1)
        .mean()
    )
    usd_change_hist['time'] = pd.Timestamp('2000-01-01', tz='UTC') + pd.to_timedelta(usd_change_hist['minute_of_day'], unit='m')

    label_user = str(TIKTOK_USERNAME) if TIKTOK_USERNAME else 'all users'
    title = (
        f'Average Change in Cumulative USD by Time of Day ({label_user}, '
        f'{USD_CHANGE_WINDOW_MINUTES}-minute moving average)'
    )
    bar_color = sns.color_palette('rocket', 1).as_hex()[0]

    fig_usd_change = px.bar(
        usd_change_hist,
        x='time',
        y='avg_delta_cumulative_usd_smoothed',
        title=title,
        template='plotly_white',
        hover_data={
            'avg_delta_cumulative_usd': ':.4f',
            'avg_delta_cumulative_usd_smoothed': ':.4f',
            'minute_of_day': True,
        },
        labels={
            'time': 'Time of Day (UTC)',
            'avg_delta_cumulative_usd_smoothed': 'Average Change in Cumulative USD (Smoothed)',
            'avg_delta_cumulative_usd': 'Average Change in Cumulative USD (Raw)',
            'minute_of_day': 'Minute of Day',
        },
    )
    fig_usd_change.update_traces(marker_color=bar_color)
    fig_usd_change.update_layout(hovermode='x unified', bargap=0)
    fig_usd_change.show()

In [21]:
if working.empty:
    print('No rows matched the selected filters.')
else:
    ARRIVAL_RATE_WINDOW_MINUTES = 30

    arrival_rate = (
        working.groupby(['day', 'minute_of_day'], as_index=False)
        .size()
        .rename(columns={'size': 'gift_events'})
    )

    all_days = working['day'].dropna().drop_duplicates().sort_values()
    minute_grid = pd.MultiIndex.from_product(
        [all_days, range(1440)],
        names=['day', 'minute_of_day'],
    ).to_frame(index=False)

    arrival_rate = minute_grid.merge(arrival_rate, on=['day', 'minute_of_day'], how='left')
    arrival_rate['gift_events'] = arrival_rate['gift_events'].fillna(0.0)

    arrival_rate_hist = (
        arrival_rate.groupby('minute_of_day', as_index=False)['gift_events']
        .mean()
        .rename(columns={'gift_events': 'avg_gifts_per_minute'})
        .sort_values('minute_of_day')
    )
    arrival_rate_hist['avg_gifts_per_minute_smoothed'] = (
        arrival_rate_hist['avg_gifts_per_minute']
        .rolling(window=ARRIVAL_RATE_WINDOW_MINUTES, center=True, min_periods=1)
        .mean()
    )
    arrival_rate_hist['time'] = pd.Timestamp('2000-01-01', tz='UTC') + pd.to_timedelta(arrival_rate_hist['minute_of_day'], unit='m')

    label_user = str(TIKTOK_USERNAME) if TIKTOK_USERNAME else 'all users'
    title = (
        f'Average Gift Arrival Rate by Time of Day ({label_user}, '
        f'{ARRIVAL_RATE_WINDOW_MINUTES}-minute moving average)'
    )
    bar_color = sns.color_palette('mako', 1).as_hex()[0]

    fig_arrival_rate = px.bar(
        arrival_rate_hist,
        x='time',
        y='avg_gifts_per_minute_smoothed',
        title=title,
        template='plotly_white',
        hover_data={
            'avg_gifts_per_minute': ':.4f',
            'avg_gifts_per_minute_smoothed': ':.4f',
            'minute_of_day': True,
        },
        labels={
            'time': 'Time of Day (UTC)',
            'avg_gifts_per_minute_smoothed': 'Average Gifts per Minute (Smoothed)',
            'avg_gifts_per_minute': 'Average Gifts per Minute (Raw)',
            'minute_of_day': 'Minute of Day',
        },
    )
    fig_arrival_rate.update_traces(marker_color=bar_color)
    fig_arrival_rate.update_layout(hovermode='x unified', bargap=0)
    fig_arrival_rate.show()